In [1]:
%pip install comet_ml -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.0/787.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
!uv pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=67dd90e5-a937-4ef8-8c02-c71417653189
To: /kaggle/working/dataset.zip
100%|████████████████████████████████████████| 356M/356M [00:05<00:00, 69.8MB/s]


In [3]:
import sys

sys.path.append('/kaggle/input/datasets/maksimbessolitsyn/')

In [4]:
%pip install comet_ml -qq

Note: you may need to restart the kernel to use updated packages.


In [5]:
import logging
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning, module=r"torch(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"torch(\.|$)")
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)

logging.getLogger("comet_ml").setLevel(logging.ERROR)

os.environ["COMET_LOGGING_CONSOLE"] = "ERROR"

In [6]:
from kaggle_secrets import UserSecretsClient
import comet_ml

user_secrets = UserSecretsClient()
COMET_API_KEY = user_secrets.get_secret("COMET_API_KEY")
comet_ml.login(api_key=COMET_API_KEY)

In [7]:
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")
SEED = 42

In [8]:
from sasrec import run_ddp_training, ExperimentConfig

In [9]:
fixed_experiment_parameters = ExperimentConfig(
    graph=ExperimentConfig.GraphConfig(
        n_layers=4,
        d_model=256,
        n_heads=4,
        dropout=0.0,
        log_q_correction=1.0,
        is_cosine_similarity=True,
    ),
    data=ExperimentConfig.DataConfig(
        vocab_size=157_162,
        max_seq_len=100,
        bos=0,
        path_interactions=PATH_INTERACTIONS,
        path_embeddings=PATH_EMBEDDINGS,
        path_artists=PATH_ARTISTS,
        core_min_interaction_per_user=5,
        test_interval_seconds=7 * 24 * 60 * 60,
        max_train_events_per_user=100,
    ),
    tau=None,
    training_dataset=None,
    test_dataset=ExperimentConfig.TestDatasetConfig(
        batch_size=32,
        device="cuda",
    ),
    optimizer=None,
    scheduler=ExperimentConfig.SchedulerConfig(
        class_name=None,
        json_args={},
    ),
    training=ExperimentConfig.TrainingConfig(
        num_epochs=15,
        grad_clip=1.0,
        eval_every=1,
        logging=True,
        comet_api_key=COMET_API_KEY,
        seed=SEED,
    ),
    evaluator=ExperimentConfig.EvaluatorConfig(
        topk=100,
    ),
)

In [10]:
from dataclasses import replace
import torch

tau = ExperimentConfig.TauConfig(
    class_name="LinearTau",
    json_args={
        "initial_tau": 0.45,
        "tau_min": None,
        "tau_max": None,
        "num_epochs": 15,
        "num_tokens_per_epoch": 4_019_032,
    },
)

training_dataset = ExperimentConfig.TrainingDatasetConfig(
    batch_size=128,
    device="cuda",
    chunk_rows=64000,
    shuffle=True,
    seed=42,
    pin_memory=True,
    uniform_negative_items=None,
    in_batch_negative_items=None,
)

optimizer = ExperimentConfig.OptimizerConfig(
    class_name="AdamW",
     json_args={
        "lr": 2e-3,
        "weight_decay": 1e-5,
    },
)

for tau_min, tau_max in [(0.04, 0.05), (0.04, 0.055), (0.045, 0.055), (0.045, 0.06)]:
    print(f"Running experiment with tau_min={tau_min} and tau_max={tau_max}...")

    tau.json_args["tau_min"] = tau_min
    tau.json_args["tau_max"] = tau_max

    for uniform, unigram in [(18_000, 12_000), (22_000, 8_000), (26_000, 4_000)]:
        run_ddp_training(
            replace(
                fixed_experiment_parameters, 
                tau=tau,
                training_dataset=replace(
                    training_dataset,
                    uniform_negative_items=uniform, 
                    in_batch_negative_items=unigram,
                ),
                optimizer=optimizer
            ),
            world_size=torch.cuda.device_count()
        )

Running experiment with tau_min=0.04 and tau_max=0.05...


Epochs: 100%|██████████| 15/15 [28:32<00:00, 114.19s/it, train_loss=7.3645]


--------------------------------
Experiment name: Linear[min=0.04,max=0.05,epochs=15]
hitrate: 0.3568
recall: 0.1219
ndcg: 0.0497
coverage: 0.4393
--------------------------------


Epochs: 100%|██████████| 15/15 [28:59<00:00, 115.98s/it, train_loss=7.1483]


--------------------------------
Experiment name: Linear[min=0.04,max=0.05,epochs=15]
hitrate: 0.3499
recall: 0.1174
ndcg: 0.0463
coverage: 0.4874
--------------------------------


...

--------------------------------
Experiment name: Linear[min=0.045,max=0.06,epochs=15]
hitrate: 0.3599
recall: 0.1234
ndcg: 0.0499
coverage: 0.4068
--------------------------------


Epochs: 100%|██████████| 15/15 [28:46<00:00, 115.08s/it, train_loss=7.5013]


--------------------------------
Experiment name: Linear[min=0.045,max=0.06,epochs=15]
hitrate: 0.3580
recall: 0.1226
ndcg: 0.0499
coverage: 0.4089
--------------------------------


Epochs: 100%|██████████| 15/15 [29:00<00:00, 116.04s/it, train_loss=6.9784]


--------------------------------
Experiment name: Linear[min=0.045,max=0.06,epochs=15]
hitrate: 0.3603
recall: 0.1235
ndcg: 0.0495
coverage: 0.3736
--------------------------------
